<a href="https://colab.research.google.com/github/hilduAgl/brain-age-eeg/blob/main/notebooks/baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature-Based Baseline

This notebook builds a simple feature-based baseline for predicting a child's age from EEG. Each recording is reduced to a small set of hand-crafted features from a single channel, and a linear regression model maps those features to age. The result is the reference point the CNN has to beat.

## Goals

- Compute a constant-age reference (always predicting the training mean/median) on the validation set
- Load channel 14 (C3-P3) for each subject and decide how to handle the zero padding at the end of recordings
- Implement Hjorth parameters (activity, mobility, complexity) and test them on synthetic signals
- Implement band-power features (delta, theta, alpha, beta) consistent with the starter implementation
- Build and save feature tables for the training and validation subjects
- Explore how each feature relates to age
- Train a standardized linear regression model on the training set only
- Evaluate on the validation set using MAE and RMSE, and compare with the constant-age reference

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "/content/drive/MyDrive/BrainAge/data"
EEG_DIR = os.path.join(DATA_DIR, "pre_processed")
METADATA_PATH = os.path.join(DATA_DIR, "anon_codex_age_sex.csv")

TRAIN_PATH = os.path.join(DATA_DIR, "segments_h1k_train.pt")
VAL_PATH = os.path.join(DATA_DIR, "segments_h1k_val.pt")
TEST_PATH = os.path.join(DATA_DIR, "segments_h1k_test.pt")

RESAMPLING_RATE = 64   # Hz, as in eda.ipynb
CHANNEL = 14           # C3-P3, chosen in eda.ipynb

In [3]:
print(os.listdir(DATA_DIR))
print("EEG files:", len(os.listdir(EEG_DIR)))

['anon_codex_age_sex.csv', 'segments_h1k_val.pt', 'segments_h1k_train.pt', 'segments_h1k_test.pt', 'pre_processed']
EEG files: 1034


In [4]:
train_data = torch.load(TRAIN_PATH)
val_data = torch.load(VAL_PATH)

print("Train:", len(train_data))
print("Val:", len(val_data))
print(train_data[0])

Train: 825
Val: 103
(0, 901, 0, 5.284383562, '8128DH.edf')
